# NPDES Violation Analysis

This notebook analyzes wastewater discharge violations reported under the **National Pollutant Discharge Elimination System (NPDES)** using a pre-downloaded violations dataset from EPA ECHO, enriched with facility metadata fetched from the ECHO API.

## Data Architecture

| Source | File | Used In | Purpose |
|--------|------|---------|---------|
| EPA ECHO download tool | one or more `*_NPDES_EFF_VIOLATIONS.csv` files | Data Loading | Primary violations dataset — loaded directly with pandas, one file per state |
| [ECHO API](https://echo.epa.gov/api/echo/cwa_rest_services.get_facility_info) | `facility_lookup.csv` (cached) | Facility Enrichment | Facility metadata (name, SIC code, city, state, lat/long) keyed by NPDES_ID |

Add state CSV files to `DATA_FILES` in the **Configuration** cell to expand coverage. The API cache (`facility_lookup.csv`) is keyed by `NPDES_ID` (nationally unique) and grows automatically as new states are added.

## What this notebook does

1. **Data Loading** — Reads all CSVs in `DATA_FILES`, tags each row with its source state, and concatenates into a single DataFrame.
2. **Facility Enrichment** — Looks up facility metadata from the ECHO API for each unique `NPDES_ID`, caches results to `facility_lookup.csv`, and joins to the violations DataFrame.
3. **Analysis** — Summarizes violation trends, identifies repeat offenders, and computes compliance rates over time.
4. **Visualizations** — Produces presentation-quality charts and interactive Plotly figures for exploring the data.

---
*Primary data: EPA ECHO effluent violations download — [ECHO data downloads](https://echo.epa.gov/tools/data-downloads)*

## Environment Setup

In [ ]:
# Install dependencies (safe to re-run)
import sys
!{sys.executable} -m pip install pandas requests matplotlib seaborn plotly ipywidgets --quiet

In [ ]:
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# --- Matplotlib / Seaborn defaults ---
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'legend.framealpha': 0.8,
    'font.family': 'sans-serif',
})

sns.set_theme(
    style='whitegrid',
    palette='muted',
    font_scale=1.1,
    rc={'axes.spines.top': False, 'axes.spines.right': False},
)

# --- Pandas display options ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.show_dimensions', True)

---
## Configuration

*Edit this cell to control which states are loaded and where cached files are stored. All other cells read from these variables — no hardcoded filenames anywhere else in the notebook.*

In [ ]:
import os
import re

frames = []
for filepath in DATA_FILES:
    # Parse state abbreviation from filename (e.g. "AZ_NPDES_EFF_VIOLATIONS.csv" → "AZ")
    match = re.match(r'^([A-Z]{2})_', os.path.basename(filepath))
    state = match.group(1) if match else 'UNKNOWN'

    df = pd.read_csv(filepath, low_memory=False)
    df['SOURCE_STATE'] = state
    frames.append(df)

violations = pd.concat(frames, ignore_index=True)

print(f'Loaded {len(violations):,} violation records across {violations["SOURCE_STATE"].nunique()} state(s): '
      f'{sorted(violations["SOURCE_STATE"].unique())}')
violations.head()

In [ ]:
# ── Data sources ──────────────────────────────────────────────────────────────
# Add one entry per state CSV downloaded from EPA ECHO.
# Drop files into the data/ folder — filenames must follow: {STATE_ABBR}_NPDES_EFF_VIOLATIONS.csv
DATA_FILES = [
    'data/AZ_NPDES_EFF_VIOLATIONS.csv',
    # 'data/CA_NPDES_EFF_VIOLATIONS.csv',
    # 'data/NV_NPDES_EFF_VIOLATIONS.csv',
]

# ── Cache ──────────────────────────────────────────────────────────────────────
# Facility metadata fetched from the ECHO API is written here after the first run.
FACILITY_CACHE_FILE = 'data/facility_lookup.csv'

# ── ECHO API ───────────────────────────────────────────────────────────────────
ECHO_FACILITY_ENDPOINT = 'https://echo.epa.gov/api/echo/cwa_rest_services.get_facility_info'

---
## 1. Data Loading

*Reads all CSV files listed in `DATA_FILES` (from the Configuration cell above). Each file is loaded with pandas, tagged with a `SOURCE_STATE` column parsed from its filename, then concatenated into a single `violations` DataFrame. No API calls are made in this section.*

---
## 2. Facility Enrichment

*Enrich the violations DataFrame with facility metadata from the ECHO API (`cwa_rest_services.get_facility_info`), keyed by `NPDES_ID`.*

*Workflow:*
- *Extract the list of unique `NPDES_ID` values from the loaded violations data.*
- *If `data/facility_lookup.csv` exists, load it from disk (skip API calls).*
- *Otherwise, query the ECHO API once per `NPDES_ID` for: facility name, SIC code, city, state, latitude, longitude.*
- *Save results to `data/facility_lookup.csv` for future runs.*
- *Join the facility lookup to the violations DataFrame on `NPDES_ID`.*

---
## 3. Analysis

*Compute compliance rates, identify repeat violators, and summarize trends over time.*

---
## 4. Visualizations

*Presentation-quality static charts and interactive Plotly figures.*

---
## Library Versions

In [ ]:
print('Environment ready.')
print(f'  pandas      {pd.__version__}')
print(f'  requests    {requests.__version__}')
print(f'  matplotlib  {matplotlib.__version__}')
print(f'  seaborn     {sns.__version__}')
print(f'  plotly      {plotly.__version__}')
print(f'  ipywidgets  {widgets.__version__}')